In [ ]:
import time
import functools
import jax
from typing import Callable, Any, Tuple, List, Optional

def benchmark_jax(n_repeats: int = 10, warmup: bool = True) -> Callable:
    """
    A decorator that benchmarks JAX function execution time.
    
    Args:
        n_repeats: Number of times to repeat the measurement
        warmup: Whether to perform a warmup run to trigger compilation
        
    Returns:
        A decorator function that wraps the target function
    """
    def decorator(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args, **kwargs) -> Tuple[Any, float, List[float]]:
            # Warmup run to trigger compilation
            if warmup:
                warmup_result = func(*args, **kwargs)
                _ = jax.block_until_ready(warmup_result)
            
            # Perform timed runs
            times = []
            for _ in range(n_repeats):
                start_time = time.time()
                result = func(*args, **kwargs)
                # Force synchronization
                result = jax.block_until_ready(result)
                end_time = time.time()
                times.append(end_time - start_time)
            
            avg_time = sum(times) / len(times)
            print(f"Function {func.__name__}:")
            print(f"  Average execution time over {n_repeats} runs: {avg_time:.8f} seconds")
            print(f"  Min: {min(times):.8f} seconds, Max: {max(times):.8f} seconds")
            
            return result, avg_time, times
        
        return wrapper
    
    return decorator

In [ ]:
import jax
import jax.numpy as jnp
from diffPLOG2TROE.parametrization import FallOff

rate_constant = FallOff(
    name="2CH3(+M)=C2H6(+M)",
    hpl_parameters=[6.770E+16, -1.180, 654.00],
    lpl_parameters=[3.400E+41, -7.030, 2762.00],
    falloff_parameters=[0.619, 73.20, 1180, 9999.00],
    falloff_type="troe",
    efficiencies={"H2": 2, "CO": 2, "CO2": 3, "H2O": 5},
)

T_range = jnp.array([1000]*300)
P_range = jnp.array([0.1, 0.4, 0.7, 1, 2, 5, 10, 20, 30, 40, 50, 70, 100])

# hpl = rate_constant.hpl.kinetic_constant(jnp.array([1000]*300)) # Compute the value of the HPL
# lpl = rate_constant.lpl.kinetic_constant(jnp.array([1000]*300)) # Compute the value of the LPL

@benchmark_jax(n_repeats=10, warmup=True)
def calculate_kinetic_constant(falloff_instance, T, P, composition=None):
    return falloff_instance.kinetic_constant(T, P)

result, avg_time, times = calculate_kinetic_constant(rate_constant, T_range, P_range)

In [ ]:
from jax import lax
import jax.numpy as jnp

def find_old(P_range, p_levels):
    def _find_index(p_index, i, P):
        return lax.cond(
            P <= p_levels[i],
            lambda _: i,
            lambda _: p_index,
            None,
        )

    p_index = lax.fori_loop(0, len(p_levels), lambda idx, i: _find_index(idx, i, P_range), 0)
    return p_index
    
def find_indices_efficient(P_range, p_levels):
    # Get the first insertion point where P <= p_levels[i]
    indices = jnp.searchsorted(p_levels, P_range, side="left")

    # If P is greater than all values in p_levels, set index to the last element
    indices = jnp.where(indices == len(p_levels), len(p_levels) - 1, indices)

    return indices

# Define your arrays
p_levels = jnp.array([0.01, 0.1, 1, 10, 100])
#P_range = jnp.logspace(jnp.log10(0.001), jnp.log10(1000), 8)
P_range = 1000

# Find the indices
indices = find_indices_efficient(P_range, p_levels)
print(indices)
indices = find_old(P_range, p_levels)
print(indices)

In [ ]:
import matplotlib.pyplot as plt
import jax.numpy as jnp
from diffPLOG2TROE.parametrization import FallOff
from diffPLOG2TROE.utilities.thermodynamic_utilities import calculate_effective_concentration, calculate_concentration
from diffPLOG2TROE.utilities.physical_constants import constants

In [ ]:
rate_constant = FallOff(
    name="H2O2(+M)=OH+OH(+M)",
    hpl_parameters=[2.0e+12, 0.9, 4.8749e+04],
    lpl_parameters=[2.49e+24, -2.3, 4.8749e+04],
    falloff_parameters=[0.43, 1.0e-30, 1.0e+30],
    falloff_type="troe",
    efficiencies={"H2O": 7.65, "N2": 1.5, "O2": 1.2, "HE": 0.65, "H2O2": 7.7, "H2": 3.7},
)
k, M, F = rate_constant.kinetic_constant(1000, 1)

print(k)

In [ ]:
T_range=jnp.linspace(300, 2500, 4)
P_range=jnp.logspace(jnp.log10(0.001), jnp.log10(100), 4)

M_eff = calculate_effective_concentration(1000, 1)

In [ ]:
k, M, F = rate_constant.kinetic_constant(1000, 1)

print(k)

In [ ]:
from typing import Union
from jaxtyping import Float64, Array
import jax.numpy as jnp
from diffPLOG2TROE.utilities.thermodynamic_utilities import calculate_concentration
from diffPLOG2TROE.utilities.physical_constants import constants

In [ ]:
def calculate_concentration_new(T: Union[Float64, Array], P: Union[Float64, Array]) -> Float64:
    # OLD
    # return (P / (constants.R_L_atm_K_mol * T)) * jnp.float64(0.001)

    # PEZZOTTA
    # conversion_factor = jnp.float64(0.001)
    # R = constants.R_L_atm_K_mol
    # if not (jnp.isscalar(T) or T.ndim == 0) and not (jnp.isscalar(P) or P.ndim == 0):
    #     T_grid, P_grid = jnp.meshgrid(T, P, indexing='ij')
    #     return (P_grid / (R * T_grid)) * conversion_factor
    # else:
    #     return (P / (R * T)) * conversion_factor

    T_grid, P_grid = jnp.meshgrid(T, P, indexing='ij')
    return (P_grid / (constants.R_L_atm_K_mol * T_grid)) * jnp.float64(.001)

In [ ]:
T_range=jnp.linspace(300, 2500, 1)
P_range=jnp.logspace(jnp.log10(0.001), jnp.log10(100), 300)

M = calculate_concentration(T_range, P_range)

print(M)

In [4]:
import jax.numpy as jnp

from diffPLOG2TROE.parametrization import CollisionEfficiency
from diffPLOG2TROE.parametrization.collision_efficiency import serialize_collision_efficiencies, deserialize_collision_efficiencies

In [21]:
# Create some example objects
ce1 = CollisionEfficiency({"A": 1e13, "n": 0.5, "Ea": 15000}, "H2")
ce2 = CollisionEfficiency({"A": 2e12, "n": 0.0, "Ea": 20000}, "O2")
ce3 = CollisionEfficiency(1e14, "Ar")

collision_list = [ce1, ce2, ce3]

# Approach 3: Complete serialization
serialized = serialize_collision_efficiencies(collision_list)
print(serialized)

species_list = ["H2", "H", "Ar"]
eff_values = jnp.array([serialized.get(s, {}).get('lnA', 1.0)  for s in species_list])

print(eff_values)

{'H2': {'lnA': 29.933606208922594, 'n': 0.5, 'EaR': 7548.293002481487}, 'O2': {'lnA': 28.324168296488494, 'n': 0.0, 'EaR': 10064.390669975315}, 'Ar': {'lnA': 100000000000000.0, 'n': None, 'EaR': None}}
[2.99336062e+01 1.00000000e+00 1.00000000e+14]
